<a href="https://colab.research.google.com/github/Jasonmiller513/Dissertation/blob/Updates/Africa_Test.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

**WEST AFRICA ONLY**

# STEP 1: Remove unnecessary columns

In [ ]:
import pandas as pd
import numpy as np
df = pd.read_csv('DataCoSupplyChainDataset.csv', encoding='latin1')

In [ ]:
# FILTER WEST AFRICA

west_africa_countries = [
    'Nigeria', 'Ghana', 'Senegal', 'Ivory Coast',
    'Benin', 'Togo', 'Mali', 'Burkina Faso',
    'Niger', 'Liberia', 'Sierra Leone',
    'Guinea', 'Gambia'
]

# Use Order Country
df_wa = df[df['Order Country'].isin(west_africa_countries)].copy()


In [ ]:
# List of columns to remove
columns_to_remove = [

    # Identifier columns
    'Customer Id',
    'Product Card Id',
    'Category Id',
    'Department Id',

    # Personal/customer detail columns
    'Customer Email',
    'Customer Password',
    'Customer Fname',
    'Customer Lname',
    'Customer Street',
    'Customer Zipcode',
    'Customer Full Name',

    # High-cardinality text columns
    'Product Description',
    'Product Image',
    'Product Name',


    # Duplicate or Redundant Geographic Columns
'Order Region',
'Customer Segment',
'Order City',
'Order State',
]

# Remove only columns that actually exist in dataframe
existing_columns_to_remove = [
    col for col in columns_to_remove if col in df_wa.columns
]

# Drop columns
df_rl = df_wa.drop(columns=existing_columns_to_remove)

# Display remaining columns
print("Remaining Columns:")
print(df_rl.columns.tolist())

# Display shape after removal
print("\nNew Dataset Shape:")
print(df_rl.shape)

Remaining Columns:
['Type', 'Days for shipping (real)', 'Days for shipment (scheduled)', 'Benefit per order', 'Sales per customer', 'Delivery Status', 'Late_delivery_risk', 'Category Name', 'Customer City', 'Customer Country', 'Customer State', 'Department Name', 'Latitude', 'Longitude', 'Market', 'Order Country', 'Order Customer Id', 'order date (DateOrders)', 'Order Id', 'Order Item Cardprod Id', 'Order Item Discount', 'Order Item Discount Rate', 'Order Item Id', 'Order Item Product Price', 'Order Item Profit Ratio', 'Order Item Quantity', 'Sales', 'Order Item Total', 'Order Profit Per Order', 'Order Status', 'Order Zipcode', 'Product Category Id', 'Product Price', 'Product Status', 'shipping date (DateOrders)', 'Shipping Mode']

New Dataset Shape:
(3151, 36)


In [ ]:
df = df_rl

print(df.shape)
missing_values = df.isnull().sum()
print(missing_values[missing_values > 0])

(3151, 36)
Order Zipcode    3151
dtype: int64


In [ ]:
duplicate_count = df.duplicated().sum()
print(f"Duplicate Rows: {duplicate_count}")

Duplicate Rows: 0


In [ ]:
numeric_cols = df.select_dtypes(include=np.number).columns

# Display summary statistics
print(df[numeric_cols].describe())

       Days for shipping (real)  Days for shipment (scheduled)  \
count               3151.000000                    3151.000000   
mean                   3.535386                       2.996826   
std                    1.624007                       1.366256   
min                    0.000000                       0.000000   
25%                    2.000000                       2.000000   
50%                    3.000000                       4.000000   
75%                    5.000000                       4.000000   
max                    6.000000                       4.000000   

       Benefit per order  Sales per customer  Late_delivery_risk     Latitude  \
count        3151.000000         3151.000000          3151.00000  3151.000000   
mean           21.990590          177.090132             0.52047    30.091500   
std            95.206297          102.043614             0.49966     9.773779   
min          -963.849976            7.490000             0.00000    17.982491   


# STEP 2: Feature engineering

In [ ]:
# DATE FEATURE ENGINEERING


# Convert order date to datetime
df['order date (DateOrders)'] = pd.to_datetime(
    df['order date (DateOrders)'],
    errors='coerce'
)

# Extract temporal features
df['order_year'] = df['order date (DateOrders)'].dt.year
df['order_month'] = df['order date (DateOrders)'].dt.month
df['order_day'] = df['order date (DateOrders)'].dt.day
df['order_dayofweek'] = df['order date (DateOrders)'].dt.dayofweek
df['order_weekofyear'] = df['order date (DateOrders)'].dt.isocalendar().week.astype(int)
df['is_weekend'] = df['order_dayofweek'].isin([5,6]).astype(int)

In [ ]:
# PROFIT ENGINEERING


# Create profit margin percentage
df['profit_margin'] = (
    df['Order Profit Per Order'] /
    (df['Sales'] + 1e-6)
)

# Revenue per item
df['revenue_per_item'] = (
    df['Sales'] /
    (df['Order Item Quantity'] + 1e-6)
)


# SHIPPING ENGINEERING


# Shipping urgency score
# Lower scheduled days = more urgent
df['shipping_urgency'] = (
    1 / (df['Days for shipment (scheduled)'] + 1)
)

# Encode shipping speed categories
shipping_map = {
    'Same Day': 4,
    'First Class': 3,
    'Second Class': 2,
    'Standard Class': 1
}

df['shipping_mode_encoded'] = (
    df['Shipping Mode']
    .map(shipping_map)
    .fillna(0)
)

In [ ]:
# PRODUCT / ORDER ENGINEERING


# Total items ordered
df['total_order_value_per_quantity'] = (
    df['Order Item Total'] /
    (df['Order Item Quantity'] + 1e-6)
)

# Large order flag
df['large_order'] = (
    df['Order Item Quantity'] >
    df['Order Item Quantity'].median()
).astype(int)

# High profit order flag
df['high_profit_order'] = (
    df['Order Profit Per Order'] >
    df['Order Profit Per Order'].median()
).astype(int)

In [ ]:
# GEOGRAPHIC FEATURES


# Combine market and region
if 'Market' in df.columns and 'Order Region' in df.columns:
    df['market_region'] = (
        df['Market'].astype(str) + '_' +
        df['Order Region'].astype(str)
    )

In [ ]:
# RL REWARD FEATURE ENGINEERING


# Create normalized profit score
profit_min = df['Order Profit Per Order'].min()
profit_max = df['Order Profit Per Order'].max()

df['normalized_profit'] = (
    (df['Order Profit Per Order'] - profit_min) /
    (profit_max - profit_min + 1e-6)
)

# Simulated lateness proxy
# (Since actual lateness removed to avoid leakage)
# Higher scheduled days may indicate harder shipments

df['lateness_risk_proxy'] = (
    df['Days for shipment (scheduled)'] /
    df['Days for shipment (scheduled)'].max()
)


# ENCODE CATEGORICAL VARIABLES


categorical_cols = [
    'Shipping Mode',
    'Market',
    'Order Region',
    'Category Name',
    'Order Country'
]

# Label encode categoricals
for col in categorical_cols:
    if col in df.columns:
        df[col] = df[col].astype('category').cat.codes

# Encode combined feature if created
if 'market_region' in df.columns:
    df['market_region'] = (
        df['market_region']
        .astype('category')
        .cat.codes
    )

# STEP 3: Encoding categorical variables

In [ ]:
from sklearn.preprocessing import LabelEncoder

In [ ]:
# IDENTIFY CATEGORICAL COLUMNS


categorical_columns = df.select_dtypes(include=['object']).columns

print("\nCategorical Columns:")
print(categorical_columns)


Categorical Columns:
Index(['Type', 'Delivery Status', 'Customer City', 'Customer Country',
       'Customer State', 'Department Name', 'Order Status',
       'shipping date (DateOrders)'],
      dtype='object')


In [ ]:
# LABEL ENCODING


# Store encoders if needed later
label_encoders = {}

for col in categorical_columns:

    # Initialize encoder
    le = LabelEncoder()

    # Convert to string to avoid issues
    df[col] = df[col].astype(str)

    # Fit and transform
    df[col] = le.fit_transform(df[col])

    # Save encoder
    label_encoders[col] = le

    print(f"Encoded: {col}")


# VERIFY RESULTS


print("\nEncoded Dataset Preview:")
print(df.head())

print("\nDataset Info:")
print(df.info())

Encoded: Type
Encoded: Delivery Status
Encoded: Customer City
Encoded: Customer Country
Encoded: Customer State
Encoded: Department Name
Encoded: Order Status
Encoded: shipping date (DateOrders)

Encoded Dataset Preview:
     Type  Days for shipping (real)  Days for shipment (scheduled)  \
59      2                         2                              2   
61      2                         5                              2   
201     3                         4                              4   
202     3                         4                              4   
208     3                         5                              4   

     Benefit per order  Sales per customer  Delivery Status  \
59         -184.779999          263.970001                3   
61           -3.650000           63.990002                1   
201          90.000000          200.000000                2   
202          -2.190000          156.759995                2   
208          74.089996          227.960007 

# STEP 4: Train/test temporal split

In [ ]:
f = df.sort_values(
    by='shipping date (DateOrders)'
).reset_index(drop=True)


In [ ]:
# TEMPORAL TRAIN/TEST SPLIT 80/20
split_index = int(len(df) * 0.80)

train_df = df.iloc[:split_index]
test_df = df.iloc[split_index:]

In [ ]:
print(f"Training Rows: {len(train_df)}")
print(f"Testing Rows : {len(test_df)}")

Training Rows: 2520
Testing Rows : 631


In [ ]:
print("\nTraining Date Range:")
print(
    train_df['shipping date (DateOrders)'].min(),
    "to",
    train_df['shipping date (DateOrders)'].max()
)

print("\nTesting Date Range:")
print(
    test_df['shipping date (DateOrders)'].min(),
    "to",
    test_df['shipping date (DateOrders)'].max()
)



Training Date Range:
0 to 1039

Testing Date Range:
4 to 1038


In [ ]:
# CREATE RL STATE MATRICES
state_columns = [
    'Market',
    'Order Region',
    'Category Name',
    'Order Item Quantity',
    'Sales',
    'Days for shipping (scheduled)',
    'shipping_mode_encoded'
]

# Keep only columns that exist
state_columns = [
    col for col in state_columns
    if col in df.columns
]

# Training states
X_train = train_df[state_columns].values

# Testing states
X_test = test_df[state_columns].values

print("\nTraining State Shape:")
print(X_train.shape)

print("\nTesting State Shape:")
print(X_test.shape)


Training State Shape:
(2520, 5)

Testing State Shape:
(631, 5)


# STEP 4: Normalization/scaling ONLY FOR TRAINING DATA!!!!

In [ ]:
from sklearn.preprocessing import MinMaxScaler
import joblib

In [ ]:


print("Training Shape:", train_df.shape)
print("Testing Shape :", test_df.shape)

Training Shape: (2520, 51)
Testing Shape : (631, 51)


In [ ]:
# EXCLUDE TARGET / REWARD COLUMNS
numeric_columns = train_df.select_dtypes(
    include=['int64', 'float64']
).columns

print("\nNumeric Columns:")
print(numeric_columns.tolist())
exclude_columns = [
    'Order Profit Per Order'
]

scale_columns = [
    col for col in numeric_columns
    if col not in exclude_columns
]

print("\nColumns Being Scaled:")
print(scale_columns)


Numeric Columns:
['Type', 'Days for shipping (real)', 'Days for shipment (scheduled)', 'Benefit per order', 'Sales per customer', 'Delivery Status', 'Late_delivery_risk', 'Customer City', 'Customer Country', 'Customer State', 'Department Name', 'Latitude', 'Longitude', 'Order Customer Id', 'Order Id', 'Order Item Cardprod Id', 'Order Item Discount', 'Order Item Discount Rate', 'Order Item Id', 'Order Item Product Price', 'Order Item Profit Ratio', 'Order Item Quantity', 'Sales', 'Order Item Total', 'Order Profit Per Order', 'Order Status', 'Order Zipcode', 'Product Category Id', 'Product Price', 'Product Status', 'shipping date (DateOrders)', 'order_weekofyear', 'is_weekend', 'profit_margin', 'revenue_per_item', 'shipping_urgency', 'shipping_mode_encoded', 'total_order_value_per_quantity', 'large_order', 'high_profit_order', 'normalized_profit', 'lateness_risk_proxy']

Columns Being Scaled:
['Type', 'Days for shipping (real)', 'Days for shipment (scheduled)', 'Benefit per order', 'Sal

In [ ]:
scaler = MinMaxScaler()

# Fit ONLY on training data
scaler.fit(train_df[scale_columns])

/usr/local/lib/python3.12/dist-packages/sklearn/utils/_array_api.py:776: RuntimeWarning: All-NaN slice encountered
  return xp.asarray(numpy.nanmin(X, axis=axis))
/usr/local/lib/python3.12/dist-packages/sklearn/utils/_array_api.py:793: RuntimeWarning: All-NaN slice encountered
  return xp.asarray(numpy.nanmax(X, axis=axis))


MinMaxScaler()

In [ ]:
train_df[scale_columns] = scaler.transform(
    train_df[scale_columns]
)

test_df[scale_columns] = scaler.transform(
    test_df[scale_columns]
)

/tmp/ipykernel_6577/3891669443.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  train_df[scale_columns] = scaler.transform(
/tmp/ipykernel_6577/3891669443.py:5: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  test_df[scale_columns] = scaler.transform(


In [ ]:
print("\nScaled Training Dataset Preview:")
print(train_df.head())

print("\nScaled Testing Dataset Preview:")
print(test_df.head())


Scaled Training Dataset Preview:
         Type  Days for shipping (real)  Days for shipment (scheduled)  \
59   0.666667                  0.333333                            0.5   
61   0.666667                  0.833333                            0.5   
201  1.000000                  0.666667                            1.0   
202  1.000000                  0.666667                            1.0   
208  1.000000                  0.833333                            1.0   

     Benefit per order  Sales per customer  Delivery Status  \
59            0.651135            0.520814         1.000000   
61            0.802521            0.114730         0.333333   
201           0.880792            0.390915         0.666667   
202           0.803741            0.303111         0.666667   
208           0.867495            0.447691         0.333333   

     Late_delivery_risk  Category Name  Customer City  Customer Country  ...  \
59                  0.0              4       0.130435         

In [ ]:
joblib.dump(
    scaler,
    'minmax_scaler.pkl'
)

print("Scaler saved as minmax_scaler.pkl")

Scaler saved as minmax_scaler.pkl
